In [23]:
from __future__ import annotations

import json
from pathlib import Path
from typing import List, Optional
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

# Global container for all chunked documents
all_docs: List[Document] = []

In [24]:
#  Helper to infer metadata
def infer_doc_metadata(file_path: str | Path) -> dict:
    """
    Infer useful metadata from markdown filename/path.
    You can customize these rules based on your project docs.
    """
    path = Path(file_path)
    file_name = path.name.lower()
    folder_name = path.parent.name.lower()

    # default metadata
    metadata = {
        "source": path.name,
        "file_name": path.name,
        "source_type": "markdown",
        "doc_type": "general",
        "app": "hydrology_app",
        "module": "general",
        "topic": "general",
        "section": folder_name,
    }

    # file-based rules
    if "flood" in file_name:
        metadata["module"] = "flood_susceptibility"
        metadata["topic"] = "flood"
    elif "rain" in file_name or "rainfall" in file_name:
        metadata["module"] = "rainfall_prediction"
        metadata["topic"] = "rainfall"
    elif "agent" in file_name or "chatbot" in file_name:
        metadata["module"] = "ai_agent"
        metadata["topic"] = "llm"
    elif "vector" in file_name or "rag" in file_name or "retrieval" in file_name:
        metadata["module"] = "knowledge_base"
        metadata["topic"] = "retrieval"
    elif "ui" in file_name or "streamlit" in file_name or "frontend" in file_name:
        metadata["module"] = "ui"
        metadata["topic"] = "frontend"
    elif "api" in file_name or "backend" in file_name:
        metadata["module"] = "backend"
        metadata["topic"] = "api"
    elif "model" in file_name:
        metadata["module"] = "ml_model"
        metadata["topic"] = "modeling"

    # doc type rules
    if "walkthrough" in file_name:
        metadata["doc_type"] = "walkthrough"
    elif "report" in file_name:
        metadata["doc_type"] = "report"
    elif "architecture" in file_name:
        metadata["doc_type"] = "architecture"
    elif "workflow" in file_name:
        metadata["doc_type"] = "workflow"
    elif "setup" in file_name:
        metadata["doc_type"] = "setup"
    elif "readme" in file_name:
        metadata["doc_type"] = "readme"
    elif "analysis" in file_name:
        metadata["doc_type"] = "analysis"

    return metadata


In [25]:
#  Load one markdown file
def load_markdown_file(
    file_path: str,
    custom_metadata: Optional[dict] = None,
) -> List[Document]:
    """
    Load a single markdown file and attach metadata.
    """
    loader = TextLoader(file_path, encoding="utf-8")
    docs = loader.load()

    base_metadata = infer_doc_metadata(file_path)

    if custom_metadata:
        base_metadata.update(custom_metadata)

    for doc in docs:
        doc.metadata.update(base_metadata)

    return docs

In [26]:
#  Load all markdown files from folder
def load_markdown_folder(
    folder_path: str,
    app_name: str = "hydrology_app",
) -> List[Document]:
    """
    Load all markdown files recursively from a folder.
    """
    folder_docs: List[Document] = []
    folder = Path(folder_path)

    for file in folder.rglob("*.md"):
        docs = load_markdown_file(
            str(file),
            custom_metadata={"app": app_name},
        )
        folder_docs.extend(docs)

    return folder_docs

In [27]:
#  Split + create chunk docs
def split_into_documents(
    base_doc: Document,
    chunk_size: int = 300,
    chunk_overlap: int = 50,
) -> List[Document]:

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n## ", "\n### ", "\n\n", "\n", " ", ""],
    )

    chunks = splitter.split_text(base_doc.page_content)

    chunk_docs = []

    for i, chunk in enumerate(chunks):
        new_metadata = {
            **base_doc.metadata,
            "chunk_id": i,
            "chunk_size": len(chunk),
        }

        chunk_docs.append(
            Document(
                page_content=chunk,
                metadata=new_metadata
            )
        )

    return chunk_docs


In [32]:
def build_chunk_documents(
    folder_path: str,
    app_name: str = "hydrology_app",
    chunk_size: int = 300,
    chunk_overlap: int = 50,
) -> List[Document]:
    """
    Build chunked documents from all markdown files in a folder and append to global all_docs.
    """
    global all_docs

    base_docs = load_markdown_folder(folder_path=folder_path, app_name=app_name)

    for base_doc in base_docs:
        chunk_docs = split_into_documents(
            base_doc=base_doc,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        all_docs.extend(chunk_docs)

    return all_docs

In [29]:
all_docs

[]

In [33]:
# Example usage (adjust path as needed)
docs_folder = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\rainfall_data\Books and others"

chunked_docs = build_chunk_documents(
    folder_path=docs_folder,
    app_name="hydrology_app",
    chunk_size=300,
    chunk_overlap=50,
)

print(f"Base folder: {docs_folder}")
print(f"Total chunks in global all_docs: {len(all_docs)}")

if all_docs:
    print("Sample chunk metadata:", all_docs[0].metadata)
    print("Sample chunk preview:", all_docs[0].page_content[:200])

Base folder: C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\rainfall_data\Books and others
Total chunks in global all_docs: 1248
Sample chunk metadata: {'source': 'rainfall_knowledge.md', 'file_name': 'rainfall_knowledge.md', 'source_type': 'markdown', 'doc_type': 'general', 'app': 'hydrology_app', 'module': 'rainfall_prediction', 'topic': 'rainfall', 'section': 'generated_knowledge', 'chunk_id': 0, 'chunk_size': 20}
Sample chunk preview: ## What is rainfall?


In [11]:
def build_chunk_documents_files(
    file_path: str,
    app_name: str = "hydrology_app",
    chunk_size: int = 300,
    chunk_overlap: int = 50,
) -> List[Document]:
    """
    Build chunked documents from one markdown file and append to global all_docs.
    """
    global all_docs

    base_docs = load_markdown_file(
        file_path=file_path,
        custom_metadata={"app": app_name},
    )

    for base_doc in base_docs:
        chunk_docs = split_into_documents(
            base_doc=base_doc,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        all_docs.extend(chunk_docs)

    return all_docs

In [52]:
# Example usage for one file
file_path = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\WALKTHROUGH.md"

chunked_docs = build_chunk_documents_files(
    file_path=file_path,
    app_name="hydrology_app",
    chunk_size=500,
    chunk_overlap=50,
)

print(f"Base file: {file_path}")
print(f"Total chunks in global all_docs: {len(all_docs)}")

if all_docs:
    print("Sample chunk metadata:", all_docs[0].metadata)
    print("Sample chunk preview:", all_docs[0].page_content[:200])

Base file: C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\raw\WALKTHROUGH.md
Total chunks in global all_docs: 23
Sample chunk metadata: {'source': 'walk2.md', 'file_name': 'walk2.md', 'source_type': 'markdown', 'doc_type': 'general', 'app': 'hydrology_app', 'module': 'general', 'topic': 'general', 'section': 'raw', 'chunk_id': 0, 'chunk_size': 49}
Sample chunk preview: # Hydro AI — LangChain Agent Skeleton Walkthrough


In [56]:
len(all_docs)

112

In [31]:
import os
import pickle

path = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\processed\text_data\rainfall"
os.makedirs(path, exist_ok=True)

output_file = os.path.join(path, "general_knowledge_RF.pkl")

with open(output_file, "wb") as f:
    pickle.dump(all_docs, f)

print(f"all_docs saved successfully at: {output_file}")
print(f"Total docs saved: {len(all_docs)}")

all_docs saved successfully at: C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\processed\text_data\rainfall\general_knowledge_RF.pkl
Total docs saved: 1248


In [59]:
import pickle
path = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\src\hydro_ai\data\processed\all_docs.pkl"

with open(path, "rb") as f:
    all_docs = pickle.load(f)

print(f"Loaded {len(all_docs)} documents")

Loaded 112 documents
